In [ ]:
cell = "K562"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
# Update this root_dir to your own project path before running
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:

### Generate upstream and downstream data
import os
import numpy as np
import pandas as pd

DNA_RAW_DIR = f"{root_dir}/dnabert/all_chr_pca64/"
DNA_MASK_DIR = f"{root_dir}/dnabert/all_chr_mask/"
FEAT_DIM = 64

CSV_PATH = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/{cell}_coords_{display_reso}kb.csv"
OUT_DIR = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/DNABERT/"
os.makedirs(OUT_DIR, exist_ok=True)

seg_length = 1000
n_seg_total = 100

if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"The input CSV does not exist. Please check the path：{CSV_PATH}")

df = pd.read_csv(CSV_PATH)
sample_count = len(df)
print(f"Total number of input CSV samples：{sample_count}")

unique_chrs = sorted(
    np.unique(df["chr"].values),
    key=lambda x: int(x.replace("chr", "")) if x.replace("chr", "").isdigit() else (23 if x == "chrX" else 24)
)
print(f"Chromosomes involved：{unique_chrs}")

for chrom in unique_chrs:
    print(f"\nStart processing chromosome: {chrom}")

    group_df = df[df["chr"] == chrom]
    chr_sample_count = len(group_df)
    print(f"Current number of chromosome samples：{chr_sample_count}")
    if chr_sample_count == 0:
        print(f"{chrom} No sample, skip")
        continue

    chrom_npy = os.path.join(DNA_RAW_DIR, f"{chrom}_all_1kb_feat_pca64.npy")
    chrom_mask_npy = os.path.join(DNA_MASK_DIR, f"{chrom}_mask.npy")
    if not os.path.exists(chrom_npy):
        raise FileNotFoundError(f"Chromosome DNABERT feature missing：{chrom_npy}")
    if not os.path.exists(chrom_mask_npy):
        raise FileNotFoundError(f"Chromosome mask missing：{chrom_mask_npy}")

    chrom_data = np.load(chrom_npy).astype(np.float32)
    chrom_mask_data = np.load(chrom_mask_npy).astype(np.float32)

    # Dimension verification
    if chrom_data.ndim != 2 or chrom_data.shape[1] != FEAT_DIM:
        raise ValueError(f"{chrom} feature dimension anomaly：{chrom_data.shape}，It is expected that the second dimension is{FEAT_DIM}")
    if chrom_data.shape[0] != chrom_mask_data.shape[0]:
        raise ValueError(f"{chrom} feature rows ({chrom_data.shape[0]}) != mask rows({chrom_mask_data.shape[0]})")
    print(f"load {chrom} | feat shape={chrom_data.shape}, mask shape={chrom_mask_data.shape}")

    output_windows = []
    output_mask_windows = []
    for _, bin_row in group_df.iterrows():
        curr_start = int(bin_row["start"])
        bin_end = curr_start + resolution
        center = curr_start + (bin_end - curr_start) / 2
        win_left = int(center - 50000)

        feat_list = []
        mask_list = []
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            bin_index = int(seg_l // seg_length) 

            if 0 <= bin_index < chrom_data.shape[0]:
                feat_list.append(chrom_data[bin_index])
                mask_list.append(chrom_mask_data[bin_index])
            else:
                feat_list.append(np.zeros(FEAT_DIM, dtype=np.float32))
                mask_list.append(0.0)

        output_windows.append(np.stack(feat_list, axis=0))
        output_mask_windows.append(np.array(mask_list))

    final_dna = np.stack(output_windows, axis=0)  # [N_chr,100,64]
    final_mask = np.stack(output_mask_windows, axis=0)  # [N_chr,100]

    dna_out_path = os.path.join(OUT_DIR, f"dna{cell}_{chrom}_predict_{display_reso}kb.npy")
    mask_out_path = os.path.join(OUT_DIR, f"dna{cell}_{chrom}_predict_mask_{display_reso}kb.npy")
    np.save(dna_out_path, final_dna)
    np.save(mask_out_path, final_mask)

    print(f"{chrom} Processing completed!")
    print(f"DNABERT feature array dimension：{final_dna.shape}  [N,100,{FEAT_DIM}]")
    print(f"Mask array dimension：{final_mask.shape}  [N,100]  (1 = valid base bin, 0 = N base / boundary out of bounds)")

    del group_df, chrom_data, chrom_mask_data, output_windows, output_mask_windows, final_dna, final_mask
    
print("All chromosome processing completed!")

